# 项目 3：ZTF 图像差分找移动目标

流程：两帧同夜 ZTF sci 图像（间隔 ~1.1h，myastro.io aria2 执行器下载）→
astroalign 星对齐 → 差分 → photutils 检测 +/- 残差对 → 移动目标候选。

> **吞吐实测（2026-10）**：IRSA 国际链路 ~84 KB/s，单帧 38MB ≈ 8 分钟。
> aria2 断点续传 + 4 连接就位后失败重试不从头来。
> 原计划走 ztfquery 包，但 1.28 版强制 IRSA token 认证，此处直接用匿名 ibe HTTP API。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Arial Unicode MS", "Heiti TC"]

from myastro.io.sources.ztf import ztf_frame

FRAMES = [("20200219403889", "A"), ("20200219437060", "B")]  # dt~82min
results = {}
for ffd, tag in FRAMES:
    r = ztf_frame(ffd, 673, "zg", 1, 1)
    results[tag] = r.paths[0]
    print(tag, r.paths[0].name, f"{r.paths[0].stat().st_size/1e6:.1f} MB, cache_hit={r.cache_hit}")

In [ ]:
from astropy.io import fits

def load(p):
    with fits.open(p, memmap=True) as hdul:
        return np.asarray(hdul[0].data, dtype=np.float64)

imgA, imgB = load(results["A"]), load(results["B"])
print("A/B shape:", imgA.shape)

with fits.open(results["A"]) as h:
    hdr = h[0].header
print(f"增益 {hdr.get('GAIN')} e-/DN, 视场 {hdr.get('NAXIS1')}x{hdr.get('NAXIS2')} px"
      f" @ 1.02 arcsec/px = {hdr.get('NAXIS1')*1.02/3600:.1f} deg")


## 1. 对齐（astroalign：星对配准，无需 WCS）

In [ ]:
import astroalign as aa

# 边缘裁掉 200px：ZTF 象限边缘像质差，降低误配
crop = (slice(200, -200), slice(200, -200))
aligned_B, footprint = aa.register(imgB[crop], imgA[crop])
print("对齐完成, 有效像元占比 %.1f%%" % (100*footprint.mean()))

## 2. 差分

静态恒星相消；移动目标在两帧间位移 ~30–60 px（主带小行星典型 30–60″/h × 1.1h），
差分图上留下**一正一负**两个残差。

In [ ]:
diff = aligned_B - imgA[crop]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, im, t in zip(axes, [imgA[crop], aligned_B, diff], ["A", "B(对齐)", "差分"]):
    v = np.percentile(im[np.isfinite(im)], [1, 99])
    ax.imshow(im, cmap="gray", vmin=v[0], vmax=v[1], origin="lower")
    ax.set_title(t); ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout(); fig.savefig("out/03-diff.png", dpi=150); fig

# 差分统计：恒星相消程度
ok = np.isfinite(diff)
print("差分 std = %.1f, 原图 std = %.1f → 残差/原图 = %.2f"
      % (np.std(diff[ok]), np.std(imgA[crop][ok]),
         np.std(diff[ok])/np.std(imgA[crop][ok])))

## 3. 检测 +/- 残差对 → 移动目标候选

In [ ]:
from photutils.detection import DAOStarFinder

sigma = 1.4826 * np.nanmedian(np.abs(diff[ok] - np.nanmedian(diff[ok])))
finder_pos = DAOStarFinder(fwhm=4.0, threshold=8.0*sigma)
finder_neg = DAOStarFinder(fwhm=4.0, threshold=-8.0*sigma)

pos = finder_pos(diff - np.nanmedian(diff[ok]))
neg = finder_neg(np.nanmedian(diff[ok]) - diff)   # 符号翻转找负残差
print(f"正残差 {0 if pos is None else len(pos)} 个, 负残差 {0 if neg is None else len(neg)} 个")

In [ ]:
def cols(tbl):
    xc = "x_centroid" if "x_centroid" in tbl.colnames else "xcentroid"
    yc = "y_centroid" if "y_centroid" in tbl.colnames else "ycentroid"
    return np.array([(x, y, f) for x, y, f in
                     zip(tbl[xc], tbl[yc], tbl["flux"])])

if pos is None or neg is None:
    print("无候选——降低阈值或换更长时间基线")
else:
    P, N = cols(pos), cols(neg)
    cands = []
    for x, y, f in P:
        d = np.hypot(N[:, 0]-x, N[:, 1]-y)
        j = int(np.argmin(d))
        if 20 < d[j] < 120 and f > 0 and N[j, 2] > 0:
            flux_ratio = min(f, N[j, 2]) / max(f, N[j, 2])
            if flux_ratio > 0.3:   # 真实小行星两次曝光通量同量级
                cands.append((x, y, d[j], flux_ratio))
    print(f"移动目标候选 {len(cands)} 个（距离/通量比）：")
    for c in sorted(cands, key=lambda c: -c[3])[:10]:
        print("  (%.0f, %.0f)  d=%.0f px  flux_ratio=%.2f" % c)

    fig, ax = plt.subplots(figsize=(8, 8))
    v = np.percentile(diff[ok], [1, 99])
    ax.imshow(diff, cmap="gray", vmin=v[0], vmax=v[1], origin="lower")
    if cands:
        cx, cy = np.array([(c[0], c[1]) for c in cands]).T
        ax.plot(cx, cy, "ro", mfc="none", ms=18, mew=1.5)
    ax.set_title("差分图 + 移动目标候选"); fig.tight_layout()
    fig.savefig("out/03-candidates.png", dpi=150); fig

## 4. 结论与下一步

- 全流程跑通：ibe 查询 → aria2 下载（断点续传）→ astroalign 对齐 → 差分 → DAOStarFinder 检测 → +/- 配对
- 差分残差/原图 std = 0.23（恒星基本相消）；8σ 阈值检出正残差 46 个
- **严格筛选（正通量 + 通量比>0.3）后 0 候选**——诚实结论：帧-帧朴素差分对单对 82min 基线
  灵敏度不够（PSF 变化 + 大气传输起伏吃掉暗弱小行星信号）。两个宽筛"候选"是伪迹
- **正确做法（下一步）**：下载 ZTP 官方差异图（suffix=`scimrefdiffimg.fits.fz`，
  参考图相减已由 IPAC 流水线完成），两历元差异图上移动目标直接显形，再做星历交叉验证

### 坑位记录
- ztfquery 1.28 强制 IRSA token（匿名 ibe HTTP 仍可用，`io/sources/ztf.py` 直接实现）
- ZTF sci URL：`sci/{YYYY}/{MMDD}/{fracday6}/ztf_{ffd}_{field:06d}_{filt}_c{ccd:02d}_{imgtype}_q{qid}_sciimg.fits`
- **ibe 元数据与 data 端点可能不一致**：部分 fracday 有元数据但整个目录 404（20200220 当晚），选帧前先 HEAD
- photutils 3.0 星表列名 `x_centroid`（非 `xcentroid`）
- astroalign 裁边后再配准（象限边缘像质差拖累星检测）
- IRSA 实测吞吐 ~84 KB/s，单帧 38MB ≈ 8 分钟——批量下载排夜间